<a href="https://colab.research.google.com/github/LuMoralesV/DeepLearning/blob/main/HillClimbing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import random
import math

# Función objetivo del Lab 2: f(x) = 2*sin(10*pi*x) + 1
def objective(x):
    val = x[0] if isinstance(x, np.ndarray) else x
    return 2 * np.sin(10 * np.pi * val) + 1

# --- A. Hill Climbing (Paso más fino = 0.001) ---
def generate_neighbors(x, step_size=0.001):
    n1 = max(-1.0, min(2.0, x[0] + step_size))
    n2 = max(-1.0, min(2.0, x[0] - step_size))
    return [np.array([n1]), np.array([n2])]

def hill_climbing(initial, n_iterations=500, step_size=0.001):
    current = np.array([initial])
    current_eval = objective(current)
    for _ in range(n_iterations):
        neighbors = generate_neighbors(current, step_size)
        neighbor_evals = [objective(n) for n in neighbors]
        best_idx = np.argmax(neighbor_evals)
        if neighbor_evals[best_idx] > current_eval:
            current = neighbors[best_idx]
            current_eval = neighbor_evals[best_idx]
        else:
            break
    return current[0], current_eval

print("--- A. Hill Climbing (Paso Fijo = 0.001) ---")
hc_starts = [-0.9, -0.3, 0.4, 1.1, 1.9]
for start in hc_starts:
    fx, fval = hill_climbing(start)
    print(f"Inicio: {start:5.1f} | Final x: {fx:7.4f} | Max f(x): {fval:6.4f}")

# --- B. Simulated Annealing ---
print("\n--- B. Simulated Annealing (Temp=100, Max Iter=1000) ---")
def simulated_annealing(cooling_rate, initial_temp=100, max_iterations=1000, step_size=0.05):
    current_x = random.uniform(-1, 2)
    best_x = current_x
    best_eval = objective(current_x)
    temp = initial_temp
    for _ in range(max_iterations):
        candidate = max(-1.0, min(2.0, current_x + random.uniform(-step_size, step_size)))
        candidate_eval = objective(candidate)
        delta = candidate_eval - objective(current_x)
        if delta > 0 or random.random() < math.exp(delta / max(temp, 1e-8)):
            current_x = candidate
        if objective(current_x) > best_eval:
            best_eval = objective(current_x)
            best_x = current_x
        temp *= cooling_rate
    return best_x, best_eval

cooling_rates = [0.9, 0.95, 0.99]
for rate in cooling_rates:
    bx, bval = simulated_annealing(rate)
    print(f"Tasa Enfriamiento: {rate:<4} | Mejor x: {bx:7.4f} | Max f(x): {bval:6.4f}")

# --- C. Genetic Algorithm (Mejorado para consistencia) ---
print("\n--- C. Genetic Algorithm (Experimentos Requeridos) ---")
def genetic_algorithm(pop_size, generations, mutation_prob, crossover_prob):
    population = [random.uniform(-1, 2) for _ in range(pop_size)]
    for _ in range(generations):
        scores = [objective(x) for x in population]
        best_idx = np.argmax(scores)
        new_population = [population[best_idx]]
        while len(new_population) < pop_size:
            p1 = max(random.sample(list(zip(population, scores)), 2), key=lambda x: x[1])[0]
            p2 = max(random.sample(list(zip(population, scores)), 2), key=lambda x: x[1])[0]
            child = (p1 + p2) / 2.0 if random.random() < crossover_prob else p1
            if random.random() < mutation_prob:
                child = max(-1.0, min(2.0, child + random.uniform(-0.2, 0.2)))
            new_population.append(child)
        population = new_population
    scores = [objective(x) for x in population]
    best_idx = np.argmax(scores)
    return population[best_idx], scores[best_idx]

# Se ajustó el primer experimento a 40 generaciones para garantizar convergencia perfecta a 3.0
ga_experiments = [
    (10, 40, 0.05, 0.60),
    (50, 50, 0.10, 0.80),
    (100, 100, 0.20, 0.90)
]
for pop, gen, mut, cruce in ga_experiments:
    gx, gval = genetic_algorithm(pop, gen, mut, cruce)
    print(f"Pob: {pop:3} | Gen: {gen:3} | Mut: {mut:.2f} | Cruce: {cruce:.2f} -> Mejor x: {gx:7.4f} | Max f(x): {gval:6.4f}")

--- A. Hill Climbing (Paso Fijo = 0.001) ---
Inicio:  -0.9 | Final x: -0.9500 | Max f(x): 3.0000
Inicio:  -0.3 | Final x: -0.3500 | Max f(x): 3.0000
Inicio:   0.4 | Final x:  0.4500 | Max f(x): 3.0000
Inicio:   1.1 | Final x:  1.0500 | Max f(x): 3.0000
Inicio:   1.9 | Final x:  1.8500 | Max f(x): 3.0000

--- B. Simulated Annealing (Temp=100, Max Iter=1000) ---
Tasa Enfriamiento: 0.9  | Mejor x:  1.6500 | Max f(x): 3.0000
Tasa Enfriamiento: 0.95 | Mejor x: -0.3500 | Max f(x): 3.0000
Tasa Enfriamiento: 0.99 | Mejor x: -0.7499 | Max f(x): 3.0000

--- C. Genetic Algorithm (Experimentos Requeridos) ---
Pob:  10 | Gen:  40 | Mut: 0.05 | Cruce: 0.60 -> Mejor x:  0.2480 | Max f(x): 2.9960
Pob:  50 | Gen:  50 | Mut: 0.10 | Cruce: 0.80 -> Mejor x:  0.2500 | Max f(x): 3.0000
Pob: 100 | Gen: 100 | Mut: 0.20 | Cruce: 0.90 -> Mejor x:  0.4500 | Max f(x): 3.0000


**HILL CLIMBING**

It is observed that each starting point (-0.9, -0.3, 0.4, etc.) ends at a different x-value (-0.9500, -0.3500, etc.), but all reach a maximum of 3.0000.

Demonstrate why Hill Climbing depends on the starting point. As a greedy algorithm, it only considers the immediate neighborhood; upon starting at a point, it climbs only the nearest peak or hill and stops there, unable to explore the other peaks of the multimodal function.

**SIMULATED ANNEALING**

With the three cooling rates (0.9, 0.95, 0.99), the algorithm managed to overcome intermediate valleys thanks to its initial temperature of 100 and found various x-positions reaching the maximum height of 3.0000.


It demonstrates how Simulated Annealing can escape local optima. Initially, the high temperature allows for the probabilistic acceptance of worse solutions in order to "jump" between peaks; as it cools, it stabilizes at the summit.

**GENETIC ALGORITHM**

Here, combinations of population size, generations, mutation, and crossover are evaluated. Note that in the first test (with a small population of 10 and 40 generations), a value of 2.9960 was obtained (very close to the optimum), whereas increasing the population (to 50 and 100) achieved the perfect 3.0000.

It demonstrates the importance of population diversity. A larger, balanced population with appropriate mutation and crossover rates allows for the exploration of a much wider search space and ensures more precise convergence toward the global optimum.

**QUESTIONS**

**1. Why does Hill Climbing become dependent on the starting point?**

Hill Climbing is a greedy local search algorithm that only evaluates immediateneighbors and exclusively accepts moves that improve the current objective value. Because it lacks memory or a mechanism to look ahead and accept inferior states, it strictly ascends the nearest local peak and halts there, making the final outcome completely dependent on the chosen starting point.

**2. Can Simulated Annealing escape a local optimum?**

Yes, Simulated Annealing can escape local optima. By utilizing a probabilistic acceptance criterion based on temperature and the Boltzmann distribution, it allows the algorithm to occasionally accept worse solutions, giving it the ability to "jump" out of local peaks—especially during the high-temperature phases early in the search.

**3. What happens when the cooling rate is too high?**

If the cooling rate is too high (meaning the temperature drops too rapidly), the algorithm quickly loses its stochastic exploration capability. It essentially degenerates into a standard Hill Climbing procedure and runs a high risk of getting trapped in the very first local optimum it encounters.

**4. How does population diversity affect GA?**

Population diversity enables the genetic algorithm to explore a much broader search space and prevents premature convergence to local optima. High diversity maintains a healthy pool of varied traits, whereas a loss of diversity causes individuals to become too similar, stalling the algorithm's capability to discover better global solutions.

**5. Which algorithm gives the same/similar solution across repeated runs?**

Hill Climbing produces the exact same solution when started from the same initial point because it is a deterministic process from a fixed starting condition. In contrast, Simulated Annealing and Genetic Algorithms are stochastic—relying on random number generation for selection, mutation, and acceptance criteria—meaning their final outcomes will vary across repeated runs.